
### The Complete 5-Step Coding Plan

We will structure the code cleanly across 5 distinct execution phases:

#### Phase 1: Environment Setup & Dataset Pipeline

* Load lightweight instruction dataset (e.g., `databricks-dolly-15k` subset or MiniLLM instruction sample) using Hugging Face `datasets`.


* Preprocess prompts using the instruction wrapper (Figure 9 from paper).



#### Phase 2: Teacher & Student Initialization

* Load Teacher model in `torch.float16` / `bfloat16` and freeze all weights (`requires_grad = False`).
* Load Student model in FP16/FP32 for training.
* Phase 1 SFT warm-up initialization check (as outlined in Algorithm 1).



#### Phase 3: On-Policy Mixture Sampler

* Implement step-by-step rollout sampling function:

$$\tilde{p}(y_t \mid y_{<t}, x) = \alpha \cdot p(y_t \mid y_{<t}, x) + (1 - \alpha) \cdot q_\theta(y_t \mid y_{<t}, x)$$



* Generate student completions $y \sim \tilde{p}$ on-policy.



#### Phase 4: Core MiniLLM Loss Engine

* Calculate Token-Level Reverse-KL Single-Step Reward:

$$r_t = \log p_{\text{teacher}}(y_t \mid y_{<t}, x) - \log q_{\text{student}}(y_t \mid y_{<t}, x)$$



* Implement **Single-Step Decomposition** $(\nabla \mathcal{L})_{\text{Single}}$ (exact sum over vocabulary $V$).


* Implement **Length-Normalized Long-Term Term** $(\nabla \mathcal{L})_{\text{Long}}^{\text{Norm}}$.


* Compute Importance Sampling Weights $w_t \approx \frac{q_\theta}{\tilde{p}}$ and accumulated policy gradient loss.



#### Phase 5: Training Loop & Research Diagnostic Plotter

* Execute training loop over $N$ steps, logging policy loss, mean reward, and learning rate.


* Plotting script generating:
* **Plot A:** Temperature Softening Curves ($T=1$ vs $T=5$).
* **Plot B:** Reverse-KL Reward Convergence over training steps.
* **Plot C:** Student Loss & Perplexity tracking curve.